In [2]:
pip install datasets 

  Using cached datasets-4.8.5-py3-none-any.whl.metadata (19 kB)
  Using cached huggingface_hub-1.12.0-py3-none-any.whl.metadata (14 kB)
  Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl.metadata (4.9 kB)
  Using cached dill-0.4.1-py3-none-any.whl.metadata (10 kB)
Using cached datasets-4.8.5-py3-none-any.whl (528 kB)
Using cached huggingface_hub-1.12.0-py3-none-any.whl (646 kB)
Using cached hf_xet-1.4.3-cp37-abi3-win_amd64.whl (3.7 MB)
Using cached dill-0.4.1-py3-none-any.whl (120 kB)

  Attempting uninstall: dill

    Found existing installation: dill 0.4.0

    Uninstalling dill-0.4.0:

      Successfully uninstalled dill-0.4.0

   ------------- -------------------------- 2/6 [dill]
   -------------------- ------------------- 3/6 [multiprocess]
   -------------------------- ------------- 4/6 [huggingface-hub]
   -------------------------- ------------- 4/6 [huggingface-hub]
   -------------------------- ------------- 4/6 [huggingface-hub]
   -------------------------- ------------- 

Data Acquisition

In [1]:
from datasets import load_dataset
import pandas as pd

c:\Users\fathy\anaconda3\envs\ai-engineering\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset('jacob-hugging-face/job-descriptions', split='train')

In [3]:
df= pd.DataFrame(dataset).sample(500, random_state=42)

In [4]:
df['full text'] = df['position_title'] + ' '+ df['job_description']
print(f'loaded {len(df)} jobs.')

loaded 500 jobs.


Embedding & Search Logic 

In [5]:
pip install sentence-transformers

  Using cached sentence_transformers-5.4.1-py3-none-any.whl.metadata (17 kB)
Using cached sentence_transformers-5.4.1-py3-none-any.whl (571 kB)
Note: you may need to restart the kernel to use updated packages.


In [6]:
from sentence_transformers import SentenceTransformer, util
import numpy as np

In [7]:
model = SentenceTransformer('all-MiniLM-L6-v2')

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 5722.03it/s]


In [8]:
embeddings = model.encode(df['full text'].tolist(), convert_to_numpy=True)

c:\Users\fathy\anaconda3\envs\ai-engineering\Lib\site-packages\transformers\integrations\sdpa_attention.py:92: UserWarning: 1Torch was not compiled with flash attention. (Triggered internally at C:\cb\pytorch_1000000000000\work\aten\src\ATen\native\transformers\cuda\sdp_utils.cpp:555.)
  attn_output = torch.nn.functional.scaled_dot_product_attention(


In [9]:
np.save('job_enbeddings.npy', embeddings)
df.to_csv('jobs.csv', index=False)

In [11]:
import torch

def search_jobs(query, top_k=5):
    query_embedding = model.encode(query)
    cos_scores = util.cos_sim(query_embedding, embeddings)[0]
    top_results = torch.topk(cos_scores, k=top_k)

    for score, idx in zip(top_results[0], top_results[1]):
        print(f"Score: {score:.4f} | Job: {df.iloc[int(idx)]['position_title']}")


query = input('Enter skills or job title: ')
search_jobs(query)

Score: 0.5954 | Job: Engineering Lead - Analytics Platform
Score: 0.5843 | Job: Remote Software Developer (Remote)
Score: 0.5436 | Job: Business Development Manager (Monsey)
Score: 0.5413 | Job: Project Manager
Score: 0.5297 | Job: Software Engineer - Reno, NV


In [14]:
import streamlit as st 
st.title("🚀 Semantic job Search 2026")
quary = st.text_input('what are you looking for ?', 'Remote python developer with AI experience')
if quary:
    quary_vec = model.encode(quary)
    scores = util.cos_sim(quary_vec, embeddings)[0]
    df['score'] = scores.tolist()

    results = df.sort_values(by='score', ascending=False).head(5)

    for _, row in results.iterrows():
        with st.expander(f"{row['position_title']} (Match:{row['score']:.2%})"):
            st.write(f"**company:**{row['company_name']}")
            st.write(row['position_title'][:500]+ "...")

2026-04-29 11:13:53.897 WARNING streamlit.runtime.scriptrunner_utils.script_run_context: Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 11:13:54.106 
  command:

    streamlit run c:\Users\fathy\anaconda3\envs\ai-engineering\Lib\site-packages\ipykernel_launcher.py [ARGUMENTS]
2026-04-29 11:13:54.107 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 11:13:54.107 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 11:13:54.107 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 11:13:54.108 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 11:13:54.108 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-04-29 11:13:54.108 Ses